# AI Therapist — QLoRA Fine-Tuning on Google Colab

Fine-tune **phxdev/psychology-qwen-0.5b** (Qwen2.5-0.5B + psychology LoRA) on merged counseling datasets.

> **Important:** Fine-tuning teaches communication style, not clinical competence. Crisis routing, PII scrubbing, and RAG must live in your application layer.

## Before you run
1. **Runtime → Change runtime type → GPU**
2. Run [`prepare_training_data.ipynb`](./prepare_training_data.ipynb) first to build `clinical_synthetic_data.json`
3. [Hugging Face account](https://huggingface.co/join) + access token

Run all cells **top to bottom**.

## 1. Verify GPU

In [ ]:
!nvidia-smi

import torch

assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime → Change runtime type → Hardware accelerator: GPU"
)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Install dependencies

In [ ]:
!pip install -q \
  "transformers>=4.44.0" \
  "datasets>=2.19.0" \
  "accelerate>=0.33.0" \
  "peft>=0.12.0" \
  "trl>=0.9.6" \
  "bitsandbytes>=0.43.0" \
  "sentencepiece" \
  "protobuf"

## 3. Hugging Face login

In [ ]:
from huggingface_hub import login

# Option A: paste token when prompted
login()

# Option B: Colab Secrets (key icon in sidebar) — uncomment below
# from google.colab import userdata
# login(token=userdata.get("HF_TOKEN"))

## 4. Mount Google Drive & set paths

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/ai-therapist"
DATA_PATH = f"{PROJECT_DIR}/clinical_synthetic_data.json"
OUTPUT_DIR = f"{PROJECT_DIR}/ai-therapist-lora"
FINAL_DIR = f"{PROJECT_DIR}/ai-therapist-lora-final"

!mkdir -p "{PROJECT_DIR}"
print(f"Project dir: {PROJECT_DIR}")
print(f"Data path:   {DATA_PATH}")

## 5. Verify training data

Expect `clinical_synthetic_data.json` from `prepare_training_data.ipynb`. Set `USE_SAMPLE_DATA = True` only for a quick smoke test.

In [ ]:
import json
import os

from transformers import AutoTokenizer

USE_SAMPLE_DATA = False

BASE_MODEL_ID = "Qwen/Qwen2.5-0.5B"
SYSTEM_PROMPT = (
    "You are a supportive, non-clinical listening assistant. "
    "You do not diagnose, prescribe medication, or handle mental health crises. "
    "You use active listening and gentle Socratic questions. "
    "Encourage professional help when concerns are serious or persistent."
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)


def format_qwen(user: str, assistant: str, system: str = SYSTEM_PROMPT) -> str:
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
        {"role": "assistant", "content": assistant},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )


if USE_SAMPLE_DATA or not os.path.exists(DATA_PATH):
    sample_rows = [
        format_qwen(
            "I've been feeling overwhelmed at work lately.",
            "That sounds really exhausting. What part of work feels most heavy right now?",
        ),
        format_qwen(
            "I keep replaying a conversation with my friend.",
            "It's tough when a moment keeps looping. What part bothers you most?",
        ),
    ]
    with open(DATA_PATH, "w", encoding="utf-8") as f:
        json.dump([{"text": t} for t in sample_rows], f, indent=2, ensure_ascii=False)
    print(f"Wrote {len(sample_rows)} smoke-test examples to {DATA_PATH}")
else:
    with open(DATA_PATH, encoding="utf-8") as f:
        n = len(json.load(f))
    print(f"Using prepared dataset at {DATA_PATH} ({n} rows)")

## 6. Configuration

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

BASE_MODEL_ID = "Qwen/Qwen2.5-0.5B"
PSYCHOLOGY_ADAPTER_ID = "phxdev/psychology-qwen-0.5b"

MAX_SEQ_LENGTH = 1024
LORA_R = 16
LORA_ALPHA = 32
MAX_STEPS = 300
BATCH_SIZE = 4
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4

## 7. Load model (base + psychology adapter + new LoRA)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

psych_model = PeftModel.from_pretrained(base_model, PSYCHOLOGY_ADAPTER_ID)
model = psych_model.merge_and_unload()

model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 8. Load dataset

In [ ]:
dataset = load_dataset("json", data_files=DATA_PATH, split="train")
print(f"Examples: {len(dataset)}")
print(dataset[0]["text"][:400], "...")

## 9. Train

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    save_steps=100,
    max_steps=MAX_STEPS,
    optim="paged_adamw_8bit",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none",
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)

trainer.train()

## 10. Save LoRA adapters

In [ ]:
trainer.model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"Saved LoRA adapters to {FINAL_DIR}")

## 11. (Optional) Push to Hugging Face Hub

In [ ]:
PUSH_TO_HUB = False
REPO_ID = "your-username/ai-therapist-lora"

if PUSH_TO_HUB:
    trainer.model.push_to_hub(REPO_ID)
    tokenizer.push_to_hub(REPO_ID)
    print(f"Pushed to https://huggingface.co/{REPO_ID}")
else:
    print("Skipped Hub upload.")

## 12. Inference smoke test

In [ ]:
from peft import PeftModel
import gc

del model, trainer
gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
test_model = PeftModel.from_pretrained(base_model, FINAL_DIR)
test_model.eval()

prompt = format_qwen(
    "I can't stop worrying about everything.",
    "",
)

inputs = tokenizer(prompt, return_tensors="pt").to(test_model.device)
with torch.no_grad():
    outputs = test_model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 13. (Optional) Download adapters as ZIP

In [ ]:
DOWNLOAD_ZIP = False

if DOWNLOAD_ZIP:
    import shutil
    from google.colab import files

    zip_path = "/content/ai-therapist-lora-final"
    shutil.make_archive(zip_path, "zip", FINAL_DIR)
    files.download(f"{zip_path}.zip")
else:
    print("Adapters saved on Drive:", FINAL_DIR)

---

## Next steps

1. Evaluate on `intima_eval.json` (red-team / companionship benchmark)
2. Run **DPO alignment** with clinician preference pairs
3. Deploy with crisis routing, PII scrubbing, and RAG

See `colab_training_guide.md` and `ai_fine_tune.md` in the repo.